In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True) # Reset kernel

In [ ]:
!pip install vllm==0.10.0
!pip install triton==3.2.0
!pip install langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai
!pip uninstall -y openai
!pip install openai==1.90.0

In [ ]:
# Install pyngrok
!pip install pyngrok

# Import and setup ngrok
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_KEY = user_secrets.get_secret("NGROK_KEY_2")

# Set ngrok authtoken
ngrok.set_auth_token(NGROK_KEY)

In [ ]:
import os
GPT_API_KEY = user_secrets.get_secret("GPT_API_KEY")
BRAVE_API_KEY = user_secrets.get_secret("BRAVE_API_KEY")
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")

os.environ["WEB_SEARCH_SSL"] = str(False)
os.environ["GOOGLE_SEARCH_CX"] = "9501a956284f141ab"
os.environ["GOOGLE_SEARCH_API_KEY"] = GOOGLE_API_KEY
os.environ["BRAVE_SEARCH_API_KEY"] = BRAVE_API_KEY
os.environ["GPT_API_KEY"] = GPT_API_KEY

DOMAIN = "http://13.215.102.203:8000"
NGROK_PORT = 8002


In [ ]:
# Start ngrok tunnel using pyngrok
tunnel = ngrok.connect(NGROK_PORT, "http")
public_url = tunnel.public_url

print(f"Ngrok tunnel started!")
print(f"Public URL: {public_url}")

In [ ]:
import requests
import io
import tarfile
import shutil
from typing import cast
def unpack(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_p(name: str):
    unpack(requests.get(f"{DOMAIN}/script/{name}").content, name)
def unpack_pl(*names: str):
    for name in names:
        unpack_p(name)
unpack_pl(
    "vllm_worker", "web_search", "server"
)

In [ ]:
from typing import Any
from web_search import CmdLogger, Websearch, initialize_vector_cache_client, get_vector_cache_client, route_search
from web_search.keyword_generator_vector import generate_search_keywords
from vllm_worker import prepare
from vllm_worker.vllm_engine import VLLMEngine, VLLMJobInfo
from server import *
from typing import AsyncGenerator
import uvicorn
import traceback

prepare()

In [ ]:
EMBEDDING_NAME = "intfloat/multilingual-e5-small"

In [ ]:
DOC_TEMPLATE = "[**{title}**]({url}):\n{text}\n"
SYSTEM_PROMPT = "Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất."
PROMPT_TEMPLATE = """{context_block}\nCâu hỏi:\n{question}\nCâu trả lời:"""
HYDE_TEMPLATE = """
Hãy trả lời câu hỏi sau ngắn gọn trong 100 từ, khi không có thông tin, đưa ra ví dụ.\n
Câu hỏi:\n{question}\n
Câu trả lời:
"""
SEP = "$$$"
SOURCE = "kaggle"
MODELS: list[ModelInfo] = [
    {
        "name": "Qwen 3 4B",
        "id": "Qwen/Qwen3-4B",
        "source": SOURCE,
        "streaming": True
    },
    {
        "name": "Qwen 3 4B LoRA Finetuned",
        "id": f"Qwen/Qwen3-4B{SEP}1",
        "source": SOURCE,
        "streaming": True
    }
]
MODEL_STATUS = [ModelStatus(**model, active=False, scheduled=False, active_count=0, scheduled_count=0) for model in MODELS]
CLIENT_INFO: KaggleServerInfo = {
    "name": "Testv1",
    "domain": "http://127.0.0.1:8002", # Auto change when run with ngrok
    "models": MODEL_STATUS
}
LORA_MAPPER = {
    1: {
        "lora_int_id": 1, # Same as key
        "lora_name": "Qwen Adapter v1",
        "lora_path": "/kaggle/input/qwen-lora-adapter/qwen_lora_adapter"
    }
}
PRELOAD_MODEL = "Qwen/Qwen3-4B"

In [ ]:
def set_active(model_id: str):
    print(f"[Global] Switched to model {model_id}")
    if SEP in model_id:
        model_id = model_id.split(SEP)[0]
    for model in CLIENT_INFO["models"]:
        if model["id"].startswith(model_id):
            model["active"] = True
            model["scheduled"] = False
        else:
            model["active"] = False
            model["scheduled"] = False

In [ ]:
class QueryRewrite:
    async def __call__(
        self, 
        text: str,
        params: GenerationParams
    ) -> tuple[str, str, str, list]:
        # Check if web search is enabled
        k_pages = params.get("k_pages", 0)
        k_docs = params.get("k_docs", 0)
        
        # Only search if web search is enabled (k_pages > 0)
        if k_pages == 0 or k_docs == 0:
            print(f"[QueryRewrite] Web search disabled (k_pages={k_pages}, k_docs={k_docs}), skipping all search")
            return text, text, text, []
        
        # Generate smart search strategy using keyword generator
        try:
            # Use keyword generator for smart routing
            vector_sources = []
            search_strategy = generate_search_keywords(text)
            print(f"[QueryRewrite] Search strategy: {search_strategy}")
            
            # If vector search is recommended, try vector DB first  
            if search_strategy.get("type_search") == "vector_db":
                keywords = search_strategy.get("key_word", [])
                vector_client = get_vector_cache_client()
                
                if vector_client:
                    # Use all provided school_ids and sections
                    source_groups = {}  # Group by school_id/section
                    
                    for keyword_data in keywords:
                        if isinstance(keyword_data, dict):
                            school_id = keyword_data.get("school_id", "").upper()
                            sections = keyword_data.get("section", "")  # Note: single section, not sections array
                            
                            if sections:  # Make sure we have a section
                                source_key = f"{school_id}_{sections}"
                                
                                print(f"[QueryRewrite] Searching vector DB: {school_id}/{sections}")
                                docs = vector_client.search_from_database(school_id, sections)
                                print(f"[QueryRewrite] Found {len(docs)} documents from vector DB")
                                
                                # Group documents by source
                                if docs:
                                    if source_key not in source_groups:
                                        source_groups[source_key] = {
                                            "title": f"Tìm trường ĐH-CĐ - Cốc Cốc ({school_id})",
                                            "url": "https://hoctap.coccoc.com/tim-truong-dh-cd",
                                            "texts": []
                                        }
                                    
                                    # Add all document contents to this source
                                    for doc in docs:
                                        source_groups[source_key]["texts"].append(doc["content"])
                        
                    # Convert grouped sources to vector_sources format with description
                    for source_key, source_info in source_groups.items():
                        # Combine all texts for this source with separators
                        combined_text = "\n\n".join(source_info["texts"])
                        # Create description from first 200 characters of text
                        description = combined_text[:200] + "..." if len(combined_text) > 200 else combined_text
                        
                        vector_sources.append({
                            "title": source_info["title"],
                            "url": source_info["url"], 
                            "description": description,  # Add required description field
                            "text": combined_text
                        })
                else:
                    print(f"[QueryRewrite] Vector client not available, will fallback to web search")
            else:
                print(f"[QueryRewrite] Using web search strategy")
            
            # Determine web query - only used if no vector sources
            if search_strategy.get("type_search") == "web_search":
                key_words = search_strategy.get("key_word", [text])
                if isinstance(key_words, list) and key_words:
                    web_query = key_words[0]  # Use first keyword for web search
                else:
                    web_query = text
            else:
                # Default web query for fallback
                web_query = text
            
        except Exception as e:
            print(f"[QueryRewrite] Error in keyword generation: {e}")
            # Fallback to original behavior
            vector_sources = []
            web_query = text
        
        # Keep original for question and rag
        question = text
        rag_query = text  # For RAG retrieval
        
        print(f"[QueryRewrite] Final result: {len(vector_sources)} vector sources, web_query: '{web_query}'")
        return question, web_query, rag_query, vector_sources

In [ ]:
class WebSearchWrapper:
    def __init__(self) -> None:
        self.web_search = Websearch(
            embedding_name=EMBEDDING_NAME, 
            device="cpu",
            chunk_size=1024, 
            chunk_overlap=128)
    async def start(self):
        await self.web_search.start()
    async def __call__(self, web_query: str, rag_query: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        k_pages = params.get("k_pages", 0)
        k_docs = params.get("k_docs", 0)
        domain_restrict = params.get("domain_restrict", False)
        print(f"[WS] k_pages: {k_pages} | k_docs {k_docs} | domain_restrict: {domain_restrict}")
        if k_pages == 0 or k_docs == 0:
            return [], []
        else:
            web_sources, rag_sources = await self.web_search(
                web_query=web_query, rag_query=rag_query,
                k_pages=k_pages, k_docs=k_docs,
                domain_restrict=domain_restrict, engine="brave",
                include_pdf=False, include_image=False
            )
            return web_sources, rag_sources

In [ ]:
class CustomQA:
    def __init__(self) -> None:
        self.engine = VLLMEngine(set_active)
        self.logger = CmdLogger("QA")
        self.query_rewriter = QueryRewrite()
        self.web_search = WebSearchWrapper()
    async def start(self):
        await self.web_search.start()
        # Initialize vector cache client
        initialize_vector_cache_client(DOMAIN)
        print(f"[CustomQA] Vector cache client initialized with domain: {DOMAIN}")
    async def delete(self):
        self.logger.start()
        del self.web_search
        self.logger.end("Unload")
    async def preload(self, model_id: str):
        vllm_in: VLLMJobInfo = {
            "model_id": model_id,
            "message": "Hello",
            "sampling_params": {
                "n": 1,
                "max_tokens": 16
            },
            "lora_request": None
        }
        generator = await self.engine.process(vllm_in)
        async for _ in generator:
            pass
    async def inference(self, prompt: str, request: KaggleRequest) -> AsyncGenerator[str, None]:
        full_model_id = request["model_id"]
        # Only support local VLLM models now (no API models)
        if SEP in full_model_id:
            model_id, lora_int_id = full_model_id.split(SEP)
            lora = LORA_MAPPER[int(lora_int_id)]
        else:
            model_id = full_model_id
            lora = None
        history = request.get("history", [])
        history = [{"role": "system", "content": SYSTEM_PROMPT}] + history
        info = {
            "message": prompt,
            "model_id": model_id,
            "sampling_params": request["params"],
            "lora_request": lora,
            "history": history
        }
        return await self.engine.process(info) #type:ignore
    async def pre_inference(
        self,
        model_id: str,
        text: str,
        stream_id: str,
        params: GenerationParams
    ) -> tuple[str, ModelPreOutput]:
        question, web_query, rag_query, vector_sources = await self.query_rewriter(text, params)
        
        # Check if search is disabled
        k_pages = params.get("k_pages", 0)
        k_docs = params.get("k_docs", 0)
        
        if k_pages == 0 or k_docs == 0:
            # Search completely disabled - no sources
            print(f"[CustomQA] Search disabled (k_pages={k_pages}, k_docs={k_docs}), no sources available")
            web_sources = []
            rag_sources = []
            all_sources = []
        elif vector_sources:
            # Use vector sources as web_sources (they now have description field)
            print(f"[CustomQA] Using vector DB sources: {len(vector_sources)} documents")
            web_sources = vector_sources  # Vector sources are now properly formatted as WebSource
            rag_sources = []  # Keep rag_sources empty for vector DB
            all_sources = vector_sources
        else:
            # No vector sources found, use web search
            print(f"[CustomQA] No vector sources found, using web search")
            web_sources, rag_sources = await self.web_search(web_query, rag_query, params)
            all_sources = rag_sources
        
        # Build context from all sources
        context = ""
        for doc in all_sources:
            content = DOC_TEMPLATE.format(
                title=doc["title"],
                url=doc["url"],
                text=doc["text"]
            )
            context += content
        context_block = f"Thông tin tham khảo:\n{context}\n" if context else ""
        prompt = PROMPT_TEMPLATE.format(
            context_block=context_block,
            question=question,
        )
        self.logger.start()
        pre_output: ModelPreOutput = {
            "stream_id": stream_id,
            "model_id": model_id,
            "generation_params": params,
            "web_sources": web_sources,  # Now includes vector sources with proper format
            "rag_sources": rag_sources,  # Empty for vector DB, populated for web search
            "extra_data": {
                "vector_sources_count": len(vector_sources),
                "web_sources_count": len(rag_sources) if not vector_sources else 0,
                "source_type": "disabled" if (k_pages == 0 or k_docs == 0) else ("vector_db" if vector_sources else "web_search")
            }
        }
        return prompt, pre_output

In [ ]:
import threading
def thread_task():
    ws_pipeline = CustomQA()
    REQUEST_STORAGE: dict[str, tuple[str, KaggleRequest]] = {}
    async def pre_inference_function(request: KaggleRequest) -> ModelPreOutput:

        prompt, pre_output = await ws_pipeline.pre_inference(
            request["model_id"],
            request["text"],
            request["stream_id"],
            request["params"]
        )
        print(request["params"])
        REQUEST_STORAGE[request["stream_id"]] = (prompt, request)
        return pre_output
    
    async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
        prompt, request = REQUEST_STORAGE.pop(stream_id)
        generator = await ws_pipeline.inference(prompt, request)
        return generator
    
    app = construct_app(
        server_domain=DOMAIN,
        info=CLIENT_INFO,
        pre_inference=pre_inference_function,
        inferece=inference_function,
        init_tasks=[
            ws_pipeline.start(), 
            ws_pipeline.preload(PRELOAD_MODEL)
        ]
    )
    
    uvicorn.run(app, port=NGROK_PORT)
thread = threading.Thread(target=thread_task, daemon=True)
thread.start()